In [1]:
from google.colab import drive
drive.mount('/content/drive')

ModuleNotFoundError: No module named 'google.colab'

In [ ]:
!pip install jiwer
!pip install --upgrade nltk
!pip install --upgrade pip
!pip install datasets gdown --no-cache transformers sentencepiece sacrebleu evaluate accelerate -U
!pip install --upgrade tensorflow
#!gdown 173lmjt9TgTYXFR4ySls5xkIjedDfnSH-UK7jxJSMus0
#!gdown 1kysE0WewbvcsRVKsm72v6-AgbgflGZ3hh7NoxIB6zZI

In [ ]:
import pandas as pd
import numpy as np
#data = pd.read_csv("/kaggle/input/115000-tn-oversampled/_115000_TN_Oversampled_Final_Data.csv", encoding ='utf-16')[["BEFORE","AFTER"]]
file_path = "/kaggle/input/1000-tn-oversampled/_1000_TN_Oversampled_Final_Data.xlsx"
data_original = pd.read_excel(file_path)
data = pd.read_excel(file_path)[["BEFORE","AFTER"]]
data.head()

In [ ]:
from sklearn.model_selection import train_test_split

train_df, val_df = train_test_split(data, test_size=0.2, shuffle=True, random_state=42)
train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)
import pandas as pd

# Assuming 'BEFORE' column contains string values
train_df['BEFORE'] = train_df['BEFORE'].astype(str)
train_df['AFTER'] = train_df['AFTER'].astype(str)
val_df['BEFORE'] = val_df['BEFORE'].astype(str)
val_df['AFTER'] = val_df['AFTER'].astype(str)
from datasets import Dataset

ds_train = Dataset.from_pandas(train_df)
ds_eval = Dataset.from_pandas(val_df)

In [ ]:
!pip install seaborn
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 2, figsize=(15, 5), sharey=True)

train_lengths = train_df["AFTER"].str.len()
sns.histplot(ax=axes[0], data=train_lengths, bins=10).set(xlabel="Length of training text samples")
axes[0].axvline(train_lengths.mean(), c="k", ls="--", lw=2.5, label="Mean")
axes[0].legend()

test_lengths = val_df["AFTER"].str.len()
sns.histplot(ax=axes[1], data=test_lengths, bins=10).set(xlabel="Length of test text samples")
axes[1].axvline(test_lengths.mean(), c="k", ls="--", lw=2.5, label="Mean")
axes[1].legend()

plt.show()

In [ ]:
data['BEFORE'] = data['BEFORE'].astype(str)
data['AFTER'] = data['AFTER'].astype(str)

res = {"translation":[]}
#print(type(data))
print(data.columns)
for row in range(len(data)):
  if data["BEFORE"][row] != "" and data["AFTER"][row] != "":
    res["translation"].append({"BEFORE": data["BEFORE"][row],"AFTER" :data["AFTER"][row]})

In [ ]:
from sklearn.model_selection import train_test_split
from datasets import Dataset

In [ ]:
data = Dataset.from_pandas(pd.DataFrame(res))
data

In [ ]:
data = data.train_test_split(test_size=0.2)

In [ ]:
data

In [ ]:
data["train"][1]

In [ ]:
# Load model directly
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM,Seq2SeqTrainingArguments,Seq2SeqTrainer
checkpoint = "csebuetnlp/banglat5"
tokenizer = AutoTokenizer.from_pretrained(checkpoint)
model = AutoModelForSeq2SeqLM.from_pretrained(checkpoint)

In [ ]:
source_lang = "AFTER"
target_lang = "BEFORE"
prefix = "Unnormalized Text to Normalaized Text:"

In [ ]:
def preprocess_function(examples):
    inputs = [prefix + example[source_lang] for example in examples["translation"]]
    targets = [example[target_lang] for example in examples["translation"]]
    model_inputs = tokenizer(inputs, text_target=targets, max_length=128, truncation=True)
    return model_inputs

In [ ]:
tokenized_books = data.map(preprocess_function, batched = True)

In [ ]:
!pip install evaluate sacrebleu
from transformers import DataCollatorForSeq2Seq
import evaluate
import numpy as np
metric = evaluate.load("sacrebleu")

In [ ]:
data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=checkpoint)

In [ ]:
def postprocess_text(preds, labels):
    preds = [pred.strip() for pred in preds]
    labels = [[label.strip()] for label in labels]
    return preds, labels

In [ ]:
"""
!pip install jiwer
!pip install bleurt
#!pip install rouge-score

import numpy as np
from datasets import load_metric
wer_metric = load_metric("wer")
meteor_metric = load_metric("meteor")

# Define the compute_metrics function
def compute_metrics(eval_preds):
    preds, labels = eval_preds
    if isinstance(preds, tuple):
        preds = preds[0]
    decoded_preds = trainer.tokenizer.batch_decode(preds, skip_special_tokens=True)

    labels = np.where(labels != -100, labels, trainer.tokenizer.pad_token_id)
    decoded_labels = trainer.tokenizer.batch_decode(labels, skip_special_tokens=True)

    decoded_preds, decoded_labels = postprocess_text(decoded_preds, decoded_labels)

    result = metric.compute(predictions=decoded_preds, references=decoded_labels)
    metrics = {"bleu": result["score"]}

    prediction_lens = [np.count_nonzero(pred != trainer.tokenizer.pad_token_id) for pred in preds]
    metrics["gen_len"] = np.mean(prediction_lens)

    pred_wer = wer_metric.compute(predictions=decoded_preds, references=decoded_labels)
    metrics["wer"] = pred_wer
    
    # Compute accuracy
    correct_predictions = sum(p == l for p, l in zip(decoded_preds, decoded_labels))
    total_predictions = len(decoded_preds)
    accuracy = correct_predictions / total_predictions
    metrics["accuracy"] = accuracy
    
    #metrics["training_loss"] = np.mean(training_history["training_loss"])
    #metrics["validation_loss"] = np.mean(training_history["validation_loss"])

    metrics = {k: round(v, 5) for k, v in metrics.items()}
    return metrics
    
"""

In [ ]:
# Garbage Collector - use it like gc.collect()
import gc
gc.collect()

In [ ]:
# Define a custom callback to log metrics
from transformers import  TrainerCallback

class LoggingCallback(TrainerCallback):
    def __init__(self):
        super().__init__()

    def on_epoch_end(self, args, state, control, model=None, tokenizer=None, **kwargs):
        if model is not None and tokenizer is not None:
            eval_metrics = trainer.evaluate()  # Remove eval_dataloader argument
            eval_preds = trainer.predict(tokenized_books["test"])
            eval_metrics = compute_metrics(eval_preds)
            training_history["eval_metrics"].append(eval_metrics)

            # Log metrics
            epoch_metrics = {
                "Epoch": state.epoch,
                "Training Loss": eval_metrics["eval_training_loss"],
                "Validation Loss": eval_metrics["eval_validation_loss"],
                "BLEU": eval_metrics["bleu"],
                "Gen Len": eval_metrics["gen_len"],
                "wer":eval_metrics["wer"]
            }
            self.log_df = self.log_df.append(epoch_metrics, ignore_index=True)
            self.log_df.to_csv("/content/logs/metrics_log.csv", index=False)

# Initialize logging callback
logging_callback = LoggingCallback()

In [ ]:
# Define a custom callback to log metrics
from transformers import TrainerCallback

# Initialize an empty dictionary to store training history
training_history = {"epoch": [], "training_loss": [], "validation_loss": [], "bleu": [], "gen_len": [], "wer": []}

# Define a callback to log training metrics
class LoggingCallback(TrainerCallback):
    def __init__(self):
        super().__init__()

    def on_epoch_end(self, args, state, control, model=None, tokenizer=None, **kwargs):
        if model is not None and tokenizer is not None:
            eval_metrics = trainer.evaluate()  # Remove eval_dataloader argument
            print(eval_metrics)
            eval_preds = trainer.predict(tokenized_books["test"])
            eval_metrics = compute_metrics(eval_preds)
            training_history["epoch"].append(state.epoch)
            training_history["training_loss"].append(eval_metrics["eval_training_loss"])
            training_history["validation_loss"].append(eval_metrics["eval_validation_loss"])
            training_history["bleu"].append(eval_metrics["eval_bleu"])
            training_history["gen_len"].append(eval_metrics["eval_gen_len"])
            training_history["wer"].append(eval_metrics["eval_wer"])

# Initialize logging callback
logging_callback = LoggingCallback()


In [ ]:
# ! pip install jiwer
# !pip install --upgrade nltk

import numpy as np
import jiwer
from datasets import load_metric
from nltk.translate.meteor_score import meteor_score

wer_metric = load_metric("wer")
meteor_metric = load_metric("meteor")

# Define the compute_metrics function
def compute_metrics(eval_preds):
    preds, labels = eval_preds
    if isinstance(preds, tuple):
        preds = preds[0]
    decoded_preds = trainer.tokenizer.batch_decode(preds, skip_special_tokens=True)

    labels = np.where(labels != -100, labels, trainer.tokenizer.pad_token_id)
    decoded_labels = trainer.tokenizer.batch_decode(labels, skip_special_tokens=True)

    decoded_preds, decoded_labels = postprocess_text(decoded_preds, decoded_labels)

    result = metric.compute(predictions=decoded_preds, references=decoded_labels)
    metrics = {"bleu": result["score"]}

    prediction_lens = [np.count_nonzero(pred != trainer.tokenizer.pad_token_id) for pred in preds]
    metrics["gen_len"] = np.mean(prediction_lens)

    pred_wer = wer_metric.compute(predictions=decoded_preds, references=decoded_labels)
    metrics["wer"] = pred_wer
    
    # Compute Character Error Rate (CER)
    meteor_score_val = meteor_metric.compute(predictions=decoded_preds, references=decoded_labels)["meteor"]
    metrics["meteor"] = meteor_score_val
    
    # Compute accuracy
    correct_predictions = sum(p == l for p, l in zip(decoded_preds, decoded_labels))
    total_predictions = len(decoded_preds)
    accuracy = correct_predictions / total_predictions
    metrics["accuracy"] = accuracy
    
    # metrics["training_loss"] = np.mean(training_history["training_loss"])
    # metrics["validation_loss"] = np.mean(training_history["validation_loss"])

    metrics = {k: round(v, 5) for k, v in metrics.items()}
    return metrics

In [ ]:
training_args = Seq2SeqTrainingArguments(
    output_dir="/kaggle/working/model",
    evaluation_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=128,
    per_device_eval_batch_size=128,
        save_steps=2500,
    eval_steps=2500,
    logging_steps=2500,
    weight_decay=0.01,
    save_total_limit=3,
    num_train_epochs=100,
    predict_with_generate=True,
    fp16=False,
    push_to_hub=False,
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_books["train"],
    eval_dataset=tokenized_books["test"],
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    #callbacks=[logging_callback]
)

history = trainer.train()

# Access the training history after training is complete
# print(training_history)

In [ ]:
import matplotlib.pyplot as plt

# Data from the table
epochs = list(range(1, 101))
training_loss = [3.039129, 2.960401, 2.914939, 2.847744, 2.788898, 2.720402, 2.646636, 2.578885, 2.504286, 2.440013,
                 2.374411, 2.309139, 2.255044, 2.197284, 2.13439, 2.086262, 2.022398, 1.980886, 1.928489, 1.890099,
                 1.845515, 1.812681, 1.768369, 1.727983, 1.676079, 1.640105, 1.610777, 1.587244, 1.553586, 1.52044,
                 1.485557, 1.456735, 1.430018, 1.40067, 1.370401, 1.342324, 1.317683, 1.295017, 1.276509, 1.259926,
                 1.242927, 1.227227, 1.215227, 1.200256, 1.184455, 1.168372, 1.15482, 1.142345, 1.12993, 1.118799,
                 1.10753, 1.095616, 1.084792, 1.074073, 1.064418, 1.056201, 1.047259, 1.0384, 1.030253, 1.023647,
                 1.015693, 1.007552, 1.000853, 0.993629, 0.987524, 0.980815, 0.973778, 0.968571, 0.961158, 0.955541,
                 0.950143, 0.943852, 0.938117, 0.932882, 0.927177, 0.922114, 0.916381, 0.910849, 0.905673, 0.901574,
                 0.896705, 0.892223, 0.887627, 0.882702, 0.877998, 0.874368, 0.869873, 0.865225, 0.860662, 0.857405,
                 0.852838, 0.848285, 0.844145, 0.839282, 0.834892, 0.830626, 0.827118, 0.823021, 0.819322, 0.815559,
                 0.811518, 0.808019]

validation_loss = [9.44605, 9.15876, 9.58038, 10.28956, 10.95339, 11.77278, 12.76359, 14.37335, 14.2739, 13.8198,
                    14.41025, 14.61094, 15.88895, 16.92444, 17.80276, 17.79984, 20.18404, 21.23684, 23.21058, 23.20365,
                    25.0009, 25.11765, 25.97242, 26.29301, 29.5726, 30.69548, 31.31568, 31.65568, 33.38192, 34.4338,
                    34.92203, 35.56233, 38.10697, 39.00292, 39.38915, 40.01397, 40.56982, 41.55062, 41.50525, 43.14782,
                    44.19677, 44.19972, 44.03728, 45.32822, 46.87284, 48.42626, 48.40494, 50.43921, 50.24697, 51.05575,
                    50.82138, 51.17185, 50.46308, 50.24871, 51.72165, 52.09274, 52.37378, 53.06534, 52.82477, 53.13556,
                    53.02429, 53.04264, 52.65577, 52.60761, 52.84294, 53.89185, 53.99278, 54.57801, 54.17338, 55.01496,
                    53.96759, 54.22697, 55.1968, 55.35482, 55.69763, 55.66374, 55.94933, 56.09078, 56.07957, 56.15317,
                    56.58487, 56.84787, 56.6685, 56.80492, 57.2917, 57.25613, 57.41523, 57.5657, 57.50419, 57.2718,
                    57.29951, 57.3934, 57.49447, 57.52237, 57.8619, 57.86376, 57.91417, 58.20999, 58.20999, 58.20999]

bleu = [8.90192, 9.08462, 8.80577, 8.80962, 8.62885, 8.36154, 8.20192, 8.125, 7.95192, 8.06154, 8.18846, 7.95769,
        8.08269, 7.96538, 8.02885, 7.97692, 7.84038, 8.08654, 7.85962, 7.99615, 7.93846, 8.09808, 7.95, 8, 7.78077,
        7.90577, 7.95769, 7.96923, 7.90192, 7.88462, 7.95192, 7.94808, 8.05577, 8.00192, 8.07692, 8.06731, 8.15,
        8.10577, 8.175, 8.10385, 8.05577, 8.08462, 8.15962, 8.19038, 8.18269, 8.17885, 8.18846, 8.13077, 8.18269,
        8.07885, 8.13846, 8.09038, 8.09423, 8.09423, 8.13462, 8.10962, 8.10577, 8.13077, 8.10192, 8.05385, 8.1,
        8.16538, 8.15962, 8.17115, 8.19615, 8.16154, 8.13269, 8.14615, 8.13462, 8.16538, 8.08846, 8.16346, 8.175,
        8.10385, 8.10385, 8.10577, 8.10192, 8.09615, 8.09615, 8.100, 8.06731, 8.06346, 8.08846, 8.08846, 8.04038,
        8.04423, 8.03269, 8.02692, 8.02885, 8.02885, 8.05385, 8.05962, 8.06731, 8.06731, 8.04231, 8.04231, 8.03654,
        8.03654, 8.03654, 8.03654, 8.03654, 8.03462, 8.03462, 8.03269, 8.02692, 8.02885, 8.03462, 8.03462, 8.03462]

gen_len = [1.0395, 1.06546, 1.0474, 1.01693, 0.98269, 0.94771, 0.9161, 0.8766, 0.86907, 0.89278, 0.91836, 0.92137,
           0.91986, 0.89579, 0.87961, 0.87622, 0.8386, 0.85628, 0.807, 0.81415, 0.78668, 0.7833, 0.77728, 0.77953,
           0.73777, 0.74, 0.73476, 0.72197, 0.70053, 0.6851, 0.68134, 0.67758, 0.66253, 0.65651, 0.6535, 0.64447,
           0.63732, 0.63168, 0.63318, 0.61588, 0.60534, 0.60798, 0.61324, 0.61475, 0.6091, 0.59481, 0.59744, 0.57938,
           0.57901, 0.57261, 0.57675, 0.57487, 0.57976, 0.58352, 0.57336, 0.56922, 0.56847, 0.56358, 0.56433, 0.56396,
           0.56772, 0.56509, 0.57073, 0.57261, 0.57148, 0.56283, 0.5617, 0.5553, 0.56057, 0.54891, 0.5617, 0.55982,
           0.54552, 0.54439, 0.54251, 0.54214, 0.54063, 0.53837, 0.53837, 0.53762, 0.53047, 0.52972, 0.53273, 0.53123,
           0.5237, 0.52408, 0.52408, 0.5222, 0.5237, 0.52521, 0.52521, 0.52295, 0.52257, 0.52182, 0.52069, 0.52144,
           0.52144, 0.51843, 0.51843, 0.51843]

wer = [1.0395, 1.06546, 1.0474, 1.01693, 0.98269, 0.94771, 0.9161, 0.8766, 0.86907, 0.89278, 0.91836, 0.92137,
       0.91986, 0.89579, 0.87961, 0.87622, 0.8386, 0.85628, 0.807, 0.81415, 0.78668, 0.7833, 0.77728, 0.77953,
       0.73777, 0.74, 0.73476, 0.72197, 0.70053, 0.6851, 0.68134, 0.67758, 0.66253, 0.65651, 0.6535, 0.64447,
       0.63732, 0.63168, 0.63318, 0.61588, 0.60534, 0.60798, 0.61324, 0.61475, 0.6091, 0.59481, 0.59744, 0.57938,
       0.57901, 0.57261, 0.57675, 0.57487, 0.57976, 0.58352, 0.57336, 0.56922, 0.56847, 0.56358, 0.56433, 0.56396,
       0.56772, 0.56509, 0.57073, 0.57261, 0.57148, 0.56283, 0.5617, 0.5553, 0.56057, 0.54891, 0.5617, 0.55982,
       0.54552, 0.54439, 0.54251, 0.54214, 0.54063, 0.53837, 0.53837, 0.53762, 0.53047, 0.52972, 0.53273, 0.53123,
       0.5237, 0.52408, 0.52408, 0.5222, 0.5237, 0.52521, 0.52521, 0.52295, 0.52257, 0.52182, 0.52069, 0.52144,
       0.52144, 0.51843, 0.51843, 0.51843]

# Plotting
plt.figure(figsize=(12, 6))

# Plot Training Loss
plt.subplot(3, 2, 1)
epochs = list(range(1, len(training_loss)+1))
plt.plot(epochs, training_loss, marker='o', color='b', label='Training Loss')
plt.title('Training Loss vs Epoch')
plt.xlabel('Epoch')
plt.ylabel('Training Loss')
plt.grid(True)
plt.legend()

# Plot Validation Loss
plt.subplot(3, 2, 2)
epochs = list(range(1, len(validation_loss)+1))
plt.plot(epochs, validation_loss, marker='o', color='r', label='Validation Loss')
plt.title('Validation Loss vs Epoch')
plt.xlabel('Epoch')
plt.ylabel('Validation Loss')
plt.grid(True)
plt.legend()

# Plot BLEU Score
plt.subplot(3, 2, 3)
epochs = list(range(1, len(bleu)+1))
plt.plot(epochs, bleu, marker='o', color='g', label='BLEU Score')
plt.title('BLEU Score vs Epoch')
plt.xlabel('Epoch')
plt.ylabel('BLEU Score')
plt.grid(True)
plt.legend()

# Plot Generation Length
plt.subplot(3, 2, 4)
epochs = list(range(1, len(gen_len)+1))
plt.plot(epochs, gen_len, marker='o', color='purple', label='Generation Length')
plt.title('Generation Length vs Epoch')
plt.xlabel('Epoch')
plt.ylabel('Generation Length')
plt.grid(True)
plt.legend()

# Plot Generation Length
plt.subplot(3, 2, 5)
epochs = list(range(1, len(wer)+1))
plt.plot(epochs, wer, marker='*', color='violet', label='WER')
plt.title('WER vs Epoch')
plt.xlabel('Epoch')
plt.ylabel('WER')
plt.grid(True)
plt.legend()

plt.tight_layout()
plt.show()


In [ ]:
trainer.evaluate()

In [ ]:
model_id = "TN_model"
trainer.save_model(model_id )

In [ ]:
# Sort by length
index = val_df["BEFORE"].str.len().sort_values(ascending=False).index
test_df = val_df.reindex(index)

In [ ]:
from transformers import pipeline

pipe = pipeline("text2text-generation", model=model_id, device=0)

texts = val_df["BEFORE"].tolist()
tns = pipe(texts, max_length=128, batch_size=16)

In [ ]:
tns

In [ ]:
tns = [tn["generated_text"] for tn in tns]
val_df["AFTER"] = tns
val_df = val_df.sort_index()
val_df.head()

In [ ]:
from transformers import AutoTokenizer
from transformers import AutoModelForSeq2SeqLM

In [ ]:
text = "Unnormalized Text to Normalaized Text: এক লক্ষ দশ হাজার পাঁচ বিয়োগ ঊনচল্লিশ লক্ষ এক শত একুশ সমান ছয় কোটি পঞ্চাশ লক্ষ ঊনত্রিশ"

In [ ]:
# checkpoint-2500
tokenizer = AutoTokenizer.from_pretrained("model/checkpoint-1500")
inputs = tokenizer(text, return_tensors="pt").input_ids
model = AutoModelForSeq2SeqLM.from_pretrained("model/checkpoint-1500")
outputs = model.generate(inputs, max_new_tokens=40, do_sample=True, top_k=30, top_p=0.95)
tokenizer.decode(outputs[0], skip_special_tokens=True)

In [ ]:
# checkpoint-3000
tokenizer = AutoTokenizer.from_pretrained("model/checkpoint-1000")
inputs = tokenizer(text, return_tensors="pt").input_ids
model = AutoModelForSeq2SeqLM.from_pretrained("model/checkpoint-1000")
outputs = model.generate(inputs, max_new_tokens=40, do_sample=True, top_k=30, top_p=0.95)
tokenizer.decode(outputs[0], skip_special_tokens=True)

In [ ]:
# checkpoint-3500
tokenizer = AutoTokenizer.from_pretrained("model/checkpoint-500")
inputs = tokenizer(text, return_tensors="pt").input_ids
model = AutoModelForSeq2SeqLM.from_pretrained("model/checkpoint-500")
outputs = model.generate(inputs, max_new_tokens=40, do_sample=True, top_k=30, top_p=0.95)
tokenizer.decode(outputs[0], skip_special_tokens=True)

In [ ]:
text2 = "Unnormalized Text to Normalaized Text: এগারই সেপ্টেম্বর উনিশশো ছিয়ানব্বই"

In [ ]:
# checkpoint-2500
tokenizer = AutoTokenizer.from_pretrained("model/checkpoint-1500")
inputs = tokenizer(text2, return_tensors="pt").input_ids
model = AutoModelForSeq2SeqLM.from_pretrained("model/checkpoint-1500")
outputs = model.generate(inputs, max_new_tokens=40, do_sample=True, top_k=30, top_p=0.95)
tokenizer.decode(outputs[0], skip_special_tokens=True)

In [ ]:
# checkpoint-3000
tokenizer = AutoTokenizer.from_pretrained("model/checkpoint-500")
inputs = tokenizer(text2, return_tensors="pt").input_ids
model = AutoModelForSeq2SeqLM.from_pretrained("model/checkpoint-500")
outputs = model.generate(inputs, max_new_tokens=40, do_sample=True, top_k=30, top_p=0.95)
tokenizer.decode(outputs[0], skip_special_tokens=True)

In [ ]:
# checkpoint-3500
tokenizer = AutoTokenizer.from_pretrained("model/checkpoint-1000")
inputs = tokenizer(text2, return_tensors="pt").input_ids
model = AutoModelForSeq2SeqLM.from_pretrained("model/checkpoint-1000")
outputs = model.generate(inputs, max_new_tokens=40, do_sample=True, top_k=30, top_p=0.95)
tokenizer.decode(outputs[0], skip_special_tokens=True)

In [ ]:
text3 = "Unnormalized Text to Normalaized Text: সাত লক্ষ সাতাত্তর হাজার চার শত বিয়াল্লিশ দশমিক আট আট আট আট সাত ছয় দুই শতাংশ"

In [ ]:
# checkpoint-2500
tokenizer = AutoTokenizer.from_pretrained("model/checkpoint-1500")
inputs = tokenizer(text3, return_tensors="pt").input_ids
model = AutoModelForSeq2SeqLM.from_pretrained("model/checkpoint-1500")
outputs = model.generate(inputs, max_new_tokens=40, do_sample=True, top_k=30, top_p=0.95)
tokenizer.decode(outputs[0], skip_special_tokens=True)

In [ ]:
# checkpoint-3000
tokenizer = AutoTokenizer.from_pretrained("model/checkpoint-500")
inputs = tokenizer(text3, return_tensors="pt").input_ids
model = AutoModelForSeq2SeqLM.from_pretrained("model/checkpoint-500")
outputs = model.generate(inputs, max_new_tokens=40, do_sample=True, top_k=30, top_p=0.95)
tokenizer.decode(outputs[0], skip_special_tokens=True)

In [ ]:
# checkpoint-3500
tokenizer = AutoTokenizer.from_pretrained("model/checkpoint-1000")
inputs = tokenizer(text3, return_tensors="pt").input_ids
model = AutoModelForSeq2SeqLM.from_pretrained("model/checkpoint-1000")
outputs = model.generate(inputs, max_new_tokens=40, do_sample=True, top_k=30, top_p=0.95)
tokenizer.decode(outputs[0], skip_special_tokens=True)

In [ ]:
# checkpoint-2500
text_1 = 'নয় লক্ষ নব্বই হাজার একষট্টি ভাগ ঊনচল্লিশ লক্ষ নিরানব্বই হাজার এক শত উনিশ' #"৯৯০০৬১/৩৯৯৯১১৯"
tokenizer = AutoTokenizer.from_pretrained("model/checkpoint-15000")
inputs = tokenizer(text_1, return_tensors="pt").input_ids
model = AutoModelForSeq2SeqLM.from_pretrained("model/checkpoint-15000")
outputs = model.generate(inputs, max_new_tokens=40, do_sample=True, top_k=30, top_p=0.95)
tokenizer.decode(outputs[0], skip_special_tokens=True)

In [ ]:
# checkpoint-2500
text_2 = 'চতুর্ভুজের মতো হলেও' #"চতুর্ভুজের মত হলেও" 
tokenizer = AutoTokenizer.from_pretrained("model/checkpoint-15000")
inputs = tokenizer(text_2, return_tensors="pt").input_ids
model = AutoModelForSeq2SeqLM.from_pretrained("model/checkpoint-15000")
outputs = model.generate(inputs, max_new_tokens=40, do_sample=True, top_k=30, top_p=0.95)
tokenizer.decode(outputs[0], skip_special_tokens=True)

In [ ]:
# checkpoint-2500
text_3 = 'কিন্তু কোনো কোনো ব্যক্তি'  #"কিন্তু কোন কোন ব্যক্তি"
tokenizer = AutoTokenizer.from_pretrained("model/checkpoint-15000")
inputs = tokenizer(text_3, return_tensors="pt").input_ids
model = AutoModelForSeq2SeqLM.from_pretrained("model/checkpoint-15000")
outputs = model.generate(inputs, max_new_tokens=40, do_sample=True, top_k=30, top_p=0.95)
tokenizer.decode(outputs[0], skip_special_tokens=True)

In [ ]:
# checkpoint-2500
text_4 = 'পরিবর্তনটা আসল খ্রিস্টপূর্ব' #"পরিবর্তনটা আসল খ্রিস্টপূর্ব" 
tokenizer = AutoTokenizer.from_pretrained("model/checkpoint-15000")
inputs = tokenizer(text_4, return_tensors="pt").input_ids
model = AutoModelForSeq2SeqLM.from_pretrained("model/checkpoint-15000")
outputs = model.generate(inputs, max_new_tokens=40, do_sample=True, top_k=30, top_p=0.95)
tokenizer.decode(outputs[0], skip_special_tokens=True)

In [ ]:
# checkpoint-2500
text_5 = 'শতাংশ চিহ্ন' #"%" 
tokenizer = AutoTokenizer.from_pretrained("model/checkpoint-15000")
inputs = tokenizer(text_5, return_tensors="pt").input_ids
model = AutoModelForSeq2SeqLM.from_pretrained("model/checkpoint-15000")
outputs = model.generate(inputs, max_new_tokens=40, do_sample=True, top_k=30, top_p=0.95)
tokenizer.decode(outputs[0], skip_special_tokens=True)

In [ ]:
# checkpoint-2500
text_6 = 'এক লক্ষ একত্রিশ হাজার নয় শত সপ্তম' #"১৩১৯০৭ম" 
tokenizer = AutoTokenizer.from_pretrained("model/checkpoint-15000")
inputs = tokenizer(text_6, return_tensors="pt").input_ids
model = AutoModelForSeq2SeqLM.from_pretrained("model/checkpoint-15000")
outputs = model.generate(inputs, max_new_tokens=40, do_sample=True, top_k=30, top_p=0.95)
tokenizer.decode(outputs[0], skip_special_tokens=True)

In [ ]:
# checkpoint-2500
text_7 =  'এক সাত তিন দুই ছয় নাম্বারে' #"১৭৩২৬ নাম্বারে"
tokenizer = AutoTokenizer.from_pretrained("model/checkpoint-15000")
inputs = tokenizer(text_7, return_tensors="pt").input_ids
model = AutoModelForSeq2SeqLM.from_pretrained("model/checkpoint-15000")
outputs = model.generate(inputs, max_new_tokens=40, do_sample=True, top_k=30, top_p=0.95)
tokenizer.decode(outputs[0], skip_special_tokens=True)

In [ ]:
# checkpoint-2500
text_8 = "দুই লক্ষ বিশ হাজার সাত পাউন্ড" #"₤২২০০০৭"
tokenizer = AutoTokenizer.from_pretrained("model/checkpoint-15000")
inputs = tokenizer(text_8, return_tensors="pt").input_ids
model = AutoModelForSeq2SeqLM.from_pretrained("model/checkpoint-15000")
outputs = model.generate(inputs, max_new_tokens=40, do_sample=True, top_k=30, top_p=0.95)
tokenizer.decode(outputs[0], skip_special_tokens=True)

In [ ]:
# checkpoint-2500
text_9 = "সকাল নয়টা ঊনপঞ্চাশ মিনিট" #"সকাল ৯:৪৯"  
tokenizer = AutoTokenizer.from_pretrained("model/checkpoint-15000")
inputs = tokenizer(text_9, return_tensors="pt").input_ids
model = AutoModelForSeq2SeqLM.from_pretrained("model/checkpoint-15000")
outputs = model.generate(inputs, max_new_tokens=40, do_sample=True, top_k=30, top_p=0.95)
tokenizer.decode(outputs[0], skip_special_tokens=True)

In [ ]:
# checkpoint-2500
text_10 = "উনিশশো সত্তর" #"১৯৭০" 
tokenizer = AutoTokenizer.from_pretrained("model/checkpoint-10000")
inputs = tokenizer(text_10, return_tensors="pt").input_ids
model = AutoModelForSeq2SeqLM.from_pretrained("model/checkpoint-10000")
outputs = model.generate(inputs, max_new_tokens=40, do_sample=True, top_k=30, top_p=0.95)
tokenizer.decode(outputs[0], skip_special_tokens=True)

In [ ]:
# checkpoint-2500
text_11 = "লেফটেন্যান্ট" #"লেঃ"
tokenizer = AutoTokenizer.from_pretrained("model/checkpoint-15000")
inputs = tokenizer(text_11, return_tensors="pt").input_ids
model = AutoModelForSeq2SeqLM.from_pretrained("model/checkpoint-15000")
outputs = model.generate(inputs, max_new_tokens=40, do_sample=True, top_k=30, top_p=0.95)
tokenizer.decode(outputs[0], skip_special_tokens=True)

In [ ]:
# checkpoint-2500
text_12 = "নভেম্বর চৌদ্দ থেকে নভেম্বর বাইশ" #"নভেম্বর ১৪ - নভেম্বর ২২" 
tokenizer = AutoTokenizer.from_pretrained("model/checkpoint-15000")
inputs = tokenizer(text_12, return_tensors="pt").input_ids
model = AutoModelForSeq2SeqLM.from_pretrained("model/checkpoint-15000")
outputs = model.generate(inputs, max_new_tokens=40, do_sample=True, top_k=30, top_p=0.95)
tokenizer.decode(outputs[0], skip_special_tokens=True)

In [ ]:
# checkpoint-2500
text_13 = "সাত শত থেকে এক হাজার তিন শত ছিয়াত্তর" #"৭০০-১৩৭৬" 
tokenizer = AutoTokenizer.from_pretrained("model/checkpoint-15000")
inputs = tokenizer(text_13, return_tensors="pt").input_ids
model = AutoModelForSeq2SeqLM.from_pretrained("model/checkpoint-15000")
outputs = model.generate(inputs, max_new_tokens=40, do_sample=True, top_k=30, top_p=0.95)
tokenizer.decode(outputs[0], skip_special_tokens=True)

In [ ]:
# checkpoint-2500
text_14 = "আটটা তিন মিনিট" #"৮:০৩ মিনিট" 
tokenizer = AutoTokenizer.from_pretrained("model/checkpoint-15000")
inputs = tokenizer(text_14, return_tensors="pt").input_ids
model = AutoModelForSeq2SeqLM.from_pretrained("model/checkpoint-15000")
outputs = model.generate(inputs, max_new_tokens=40, do_sample=True, top_k=30, top_p=0.95)
tokenizer.decode(outputs[0], skip_special_tokens=True)

In [ ]:
# checkpoint-2500
text_15 = "ড্র তিন শুণ্য" #"ড্র ৩-০" 
tokenizer = AutoTokenizer.from_pretrained("model/checkpoint-15000")
inputs = tokenizer(text_15, return_tensors="pt").input_ids
model = AutoModelForSeq2SeqLM.from_pretrained("model/checkpoint-15000")
outputs = model.generate(inputs, max_new_tokens=40, do_sample=True, top_k=30, top_p=0.95)
tokenizer.decode(outputs[0], skip_special_tokens=True)

In [ ]:
# checkpoint-2500
text_16 = "ঊনত্রিশ হাজার" #"২৯,০০০" 
tokenizer = AutoTokenizer.from_pretrained("model/checkpoint-15000")
inputs = tokenizer(text_16, return_tensors="pt").input_ids
model = AutoModelForSeq2SeqLM.from_pretrained("model/checkpoint-15000")
outputs = model.generate(inputs, max_new_tokens=40, do_sample=True, top_k=30, top_p=0.95)
tokenizer.decode(outputs[0], skip_special_tokens=True)

In [ ]:
# checkpoint-2500
text_17 =  "মাইনাস এক শত নব্বই" #"-১৯০"
tokenizer = AutoTokenizer.from_pretrained("model/checkpoint-15000")
inputs = tokenizer(text_17, return_tensors="pt").input_ids
model = AutoModelForSeq2SeqLM.from_pretrained("model/checkpoint-15000")
outputs = model.generate(inputs, max_new_tokens=40, do_sample=True, top_k=30, top_p=0.95)
tokenizer.decode(outputs[0], skip_special_tokens=True)